In [5]:
import json
import mysql.connector
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

# ONNX specific imports
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


def get_connection():
  return mysql.connector.connect(
      host="localhost",
      user="root",
      password="",
      database="marketdata",
  )


# ------------------------------------------
# 1. FETCH DATA FROM MYSQL WITH CONSTRAINT
# ------------------------------------------
try:
  db = get_connection()
  query = 'SELECT * FROM `labeled_flatdata` WHERE `edited` = "1"'
  df = pd.read_sql(query, db)
  print(
      f"Successfully loaded {len(df)} filtered rows (edited = '1') from MySQL"
      " database."
  )
except Exception as e:
  print(f"Database connection or query failed: {e}")
  df = pd.DataFrame()
finally:
  if "db" in locals() and db.is_connected():
    db.close()


# ------------------------------------------
# 2. PROCEED ONLY IF DATA IS LOADED
# ------------------------------------------
if df.empty:
  print(
      "DataFrame is empty. Check your database connection, credentials, and"
      " whether any rows have edited = '1'."
  )
else:
  annotation_columns = [
      "was",
      "then",
      "currently",
      "future",
      "volatility",
      "bias",
      "phase",
      "momentum_direction",
      "momentum_strength",
      "setup",
      "momentum_candle",
      "decision",
  ]

  valid_targets = [col for col in annotation_columns if col in df.columns]
  print(f"Valid target columns found: {valid_targets}\n")

  # ------------------------------------------
  # 3. INPUT FEATURES: M1 PRICE COLUMNS
  # ------------------------------------------
  feature_columns = [
      col
      for col in df.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  X = df[feature_columns].copy()
  X = X.apply(pd.to_numeric, errors="coerce")
  X = X.replace([np.inf, -np.inf], np.nan)

  # ------------------------------------------
  # 4. TRAIN, EXPORT ONNX, & SAVE CLASSES JSON
  # ------------------------------------------
  models = {}
  label_encoders = {}
  results = {}

  for target in valid_targets:
    y_series = df[target].fillna("").astype(str).str.strip()
    valid = (
        df[target].notna()
        & y_series.ne("")
        & y_series.ne("nan")
        & y_series.ne("None")
        & y_series.ne("NULL")
    )

    X_target = X.loc[valid]
    y_raw = y_series.loc[valid]

    print(f"Target '{target}': found {len(y_raw)} valid labeled rows.")

    if len(y_raw) < 2:
      print(
          f"Skipping {target}: insufficient labeled rows ({len(y_raw)} found,"
          " need at least 2)."
      )
      continue

    encoder = LabelEncoder()
    y = encoder.fit_transform(y_raw)

    if len(encoder.classes_) < 2:
      print(
          f"Skipping {target}: only one unique label found ({encoder.classes_}).\n"
      )
      continue

    class_counts = pd.Series(y).value_counts()
    stratify = y if class_counts.min() >= 2 else None
    test_size = 0.2 if len(X_target) >= 5 else 0.5

    X_train, X_test, y_train, y_test = train_test_split(
        X_target,
        y,
        test_size=test_size,
        random_state=42,
        stratify=stratify,
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                class_weight="balanced",
                n_jobs=-1,
            ),
        ),
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    models[target] = model
    label_encoders[target] = encoder

    results[target] = {
        "accuracy": accuracy,
        "training_rows": len(X_train),
        "testing_rows": len(X_test),
        "classes": list(encoder.classes_),
    }

    print(f"\n{'=' * 50}")
    print(f"TARGET: {target}")
    print(f"Accuracy: {accuracy:.2%}")

    # 1. Export ONNX Model
    try:
      initial_type = [("float_input", FloatTensorType([None, X_target.shape[1]]))]
      onnx_model = convert_sklearn(model, initial_types=initial_type)

      onnx_filename = f"model_{target}.onnx"
      with open(onnx_filename, "wb") as f:
        f.write(onnx_model.SerializeToString())

      # 2. Save corresponding class labels JSON
      classes_filename = f"classes_{target}.json"
      with open(classes_filename, "w") as f:
        json.dump(list(encoder.classes_), f)

      print(f"Successfully exported '{onnx_filename}' and '{classes_filename}'")
    except Exception as e:
      print(f"Failed to export {target}: {e}")

    print("-" * 50 + "\n")

  if results:
    results_df = pd.DataFrame(results).T
    display(results_df)

C:\Users\nishw\AppData\Local\Temp\ipykernel_25552\2049701663.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, db)


Successfully loaded 14 filtered rows (edited = '1') from MySQL database.
Valid target columns found: ['was', 'then', 'currently', 'future', 'volatility', 'bias', 'phase', 'momentum_direction', 'momentum_strength', 'setup', 'momentum_candle', 'decision']

Target 'was': found 13 valid labeled rows.

TARGET: was
Accuracy: 100.00%
Successfully exported 'model_was.onnx' and 'classes_was.json'
--------------------------------------------------

Target 'then': found 13 valid labeled rows.

TARGET: then
Accuracy: 100.00%
Successfully exported 'model_then.onnx' and 'classes_then.json'
--------------------------------------------------

Target 'currently': found 13 valid labeled rows.

TARGET: currently
Accuracy: 100.00%
Successfully exported 'model_currently.onnx' and 'classes_currently.json'
--------------------------------------------------

Target 'future': found 13 valid labeled rows.

TARGET: future
Accuracy: 100.00%
Successfully exported 'model_future.onnx' and 'classes_future.json'
-----

,accuracy,training_rows,testing_rows,classes
was,1.0,10,3,"[Pushing Down, Pushing Up]"
then,1.0,10,3,"[Push up, Pushed Up with a gap]"
currently,1.0,10,3,"[again a gap, push up, pushing down, pushing u..."
future,1.0,10,3,"[pull down form a HL to push up, push up after..."
volatility,1.0,10,3,"[good, ok, too_much]"
bias,0.666667,10,3,"[none, up]"
phase,1.0,10,3,"[none, pull, push]"
momentum_direction,0.666667,10,3,"[bearish, bullish, up]"
momentum_strength,1.0,10,3,"[none, strong]"
setup,0.666667,10,3,"[HL, MDB, none]"


In [4]:
import json
import os
import mysql.connector
import numpy as np
import onnxruntime as ort
import pandas as pd


# Helper to connect to DB for fetching new input data
def get_connection():
  return mysql.connector.connect(
      host="localhost",
      user="root",
      password="",
      database="marketdata",
  )


# List of all annotation targets
target_models = [
    "was",
    "then",
    "currently",
    "future",
    "volatility",
    "bias",
    "phase",
    "momentum_direction",
    "momentum_strength",
    "setup",
    "momentum_candle",
    "decision",
]

# ---------------------------------------------------------
# 1. FETCH RECORD FROM DATABASE ONCE
# ---------------------------------------------------------
db = get_connection()
query = "SELECT * FROM `labeled_flatdata` WHERE `id` = 10"
df_new = pd.read_sql(query, db)
db.close()

if df_new.empty:
  print("No record found with id = 18.")
else:
  print(f"Successfully loaded record ID 18 for inference.\n")

  # ---------------------------------------------------------
  # 2. EXTRACT AND FORMAT FEATURES ONCE
  # ---------------------------------------------------------
  feature_columns = [
      col
      for col in df_new.columns
      if col.startswith("M1_")
      and not col.endswith("_time")
      and col.rsplit("_", 1)[-1] in {"open", "high", "low", "close"}
  ]

  X_new = df_new[feature_columns].copy()
  X_new = X_new.apply(pd.to_numeric, errors="coerce")
  X_new = X_new.replace([np.inf, -np.inf], np.nan)

  # Convert DataFrame row to a 2D numpy array of type float32
  input_data = X_new.to_numpy(dtype=np.float32)

  print("=" * 60)
  print("BATCH PREDICTION RESULTS FOR ROW ID 18")
  print("=" * 60)

  # ---------------------------------------------------------
  # 3. LOOP THROUGH EACH MODEL AND RUN INFERENCE
  # ---------------------------------------------------------
  for target_name in target_models:
    onnx_model_path = f"model_{target_name}.onnx"
    classes_path = f"classes_{target_name}.json"

    # Check if model and class files exist (skips if training skipped it)
    if not os.path.exists(onnx_model_path) or not os.path.exists(classes_path):
      print(
          f"[{target_name.upper()}] -> Skipped (Model or class JSON file not"
          " found)."
      )
      continue

    try:
      # Load ONNX session & label classes
      session = ort.InferenceSession(onnx_model_path)
      with open(classes_path, "r") as f:
        class_labels = json.load(f)

      input_name = session.get_inputs()[0].name
      expected_feature_count = session.get_inputs()[0].shape[1]

      # Safety check on feature dimensions
      if input_data.shape[1] != expected_feature_count:
        print(
            f"[{target_name.upper()}] -> Skipped (Feature mismatch: expected"
            f" {expected_feature_count}, got {input_data.shape[1]})."
        )
        continue

      # Run Inference
      outputs = session.run(None, {input_name: input_data})
      predicted_class_idx = outputs[0][0]
      class_probabilities = outputs[1]

      predicted_label = class_labels[int(predicted_class_idx)]

      print(f"Target: {target_name}")
      print(f"  -> Predicted Label : {predicted_label}")
      print(f"  -> Probabilities   : {class_probabilities}")
      print("-" * 60)

    except Exception as e:
      print(f"[{target_name.upper()}] -> Error during inference: {e}")
      print("-" * 60)

Successfully loaded record ID 18 for inference.

BATCH PREDICTION RESULTS FOR ROW ID 18
[WAS] -> Skipped (Model or class JSON file not found).
Target: then
  -> Predicted Label : Pushed Up with a gap
  -> Probabilities   : [{0: 0.020000040531158447, 1: 0.9799999594688416}]
------------------------------------------------------------
Target: currently
  -> Predicted Label : pushing up
  -> Probabilities   : [{0: 0.06999999284744263, 1: 0.25, 2: 0.059999994933605194, 3: 0.6099998950958252, 4: 0.009999999776482582}]
------------------------------------------------------------
Target: future
  -> Predicted Label : pull down form a HL to push up
  -> Probabilities   : [{0: 0.9099999666213989, 1: 0.029999999329447746, 2: 0.059999994933605194}]
------------------------------------------------------------
Target: volatility
  -> Predicted Label : ok
  -> Probabilities   : [{0: 0.019999999552965164, 1: 0.9199999570846558, 2: 0.059999994933605194}]
-----------------------------------------------

C:\Users\nishw\AppData\Local\Temp\ipykernel_25552\2433313766.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_new = pd.read_sql(query, db)
